# 🎬 AI Video Upscaler 4K - Real-CUGAN Pro WebUI (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhSi1008/ai-video-upscaler/blob/main/notebooks/Google_Colab_RealCUGAN_4K.ipynb)

Hệ thống nâng cấp Anime từ 1080p lên **4K Ultra-HD (3840x2160 Native 2x)** sử dụng giao diện WebUI trực quan.
- **Giao diện WebUI:** Dán Link Google Drive hoặc chọn tệp video trực tiếp trên trình duyệt.
- **Tự động lưu Drive:** Video 4K hoàn chỉnh được lưu vĩnh viễn vào Google Drive (`/content/drive/MyDrive/Upscaled`).
- **Mã hóa chuyên nghiệp:** HEVC 10-bit (`p010le` / `yuv420p10le`), giữ nguyên 100% âm thanh và phụ đề mềm (.ass).
- **Tốc độ xử lý:** ~12-16 FPS trên GPU NVIDIA Tesla T4.

---

In [ ]:
# @title 🚀 BƯỚC 1: Kết Nối Google Drive & Kiểm Tra GPU T4
# @markdown Bấm nút Run (Ctrl+Enter) và chọn "Kết nối với Google Drive" khi bảng hỏi hiện lên.
from google.colab import drive
import torch
import os

drive.mount('/content/drive')

print("\n" + "="*60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ ĐÃ SẴN SÀNG GPU TỐC ĐỘ CAO: {gpu_name}")
else:
    print("⚠️ CHƯA BẬT GPU! Vào menu: Runtime -> Change runtime type -> T4 GPU")
print("="*60)

In [ ]:
# @title 📦 BƯỚC 2: Cài Đặt Môi Trường, Kích Hoạt NVENC & Đồng Bộ Mã Nguồn
# @markdown Cài đặt FFmpeg phần cứng GPU NVIDIA NVENC và tải mã nguồn mới nhất (~30 - 45s).
!apt-get update -qq && apt-get install -y ffmpeg -qq
!pip install -q gradio opencv-python gdown

# Kích hoạt FFmpeg NVIDIA NVENC tăng tốc mã hóa 4K lên gấp 4 lần
import subprocess, os
res = subprocess.run(['ffmpeg', '-encoders'], stdout=subprocess.PIPE, text=True)
if 'hevc_nvenc' not in res.stdout:
    print('⚡ Đang kích hoạt FFmpeg GPU Hardware Acceleration (NVENC)...')
    !git clone --depth 1 https://github.com/rokibulislaam/colab-ffmpeg-cuda.git -q
    !cp -r ./colab-ffmpeg-cuda/bin/. /usr/bin/ 2>/dev/null || true
    !rm -rf ./colab-ffmpeg-cuda

repo_dir = "/content/ai-video-upscaler"
if os.path.exists(repo_dir):
    %cd {repo_dir}
    !git pull
else:
    !git clone https://github.com/ThanhSi1008/ai-video-upscaler.git {repo_dir}
    %cd {repo_dir}

print("\n✅ ĐÃ ĐỒNG BỘ MÃ NGUỒN VÀ KÍCH HOẠT GPU TĂNG TỐC THÀNH CÔNG!")

In [ ]:
# @title 🌐 BƯỚC 3: Khởi Chạy Giao Diện WebUI 4K
# @markdown Bấm Run ô này, sau đó nhấp vào đường link Public Gradio (ví dụ: `https://xxxx.gradio.live`) để mở giao diện WebUI trên trình duyệt:
!python3 app.py --share